# AutoGen

AutoGen is a multi agent AI framework developed by Microsoft.
It lets you create multiple AI agents that talk to each other to solve task
==> very similar to CrewAI, but with more flexibility and control

**Simple terms to understand:**
* you can create agents (assistant, user proxy, coder, etc)
* they chat with each other automatically
* they collaborate to complete task

--- 
### Key Idea
user input ==> Agent1 ==> Agent2 ==> Agent3 ==> output

---
# Sample Program

In [ ]:
# install autogen
!pip install autogen pyautogen

In [ ]:
from autogen import AssistantAgent, UserProxyAgent
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# create Agent
assistant = AssistantAgent(
    name="assistant",
    lmm_config={"model": "gpt-4o-mini"}
)

# create user proxy (act like human)
user_proxy = UserProxyAgent(
    name="user",
    code_execution_config=False
)

user_proxy.initiate_chat(
    assistant,
    message="Explain AI in simple terms"
)

---
# Latest Version coding

In [ ]:
# install the autogen latest package
pip install -U "autogen-agentchat" "autogen-ext[openai]"

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import asyncio
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")


# Define a model client. You can use other model client that implements
# the `ChatCompletionClient` interface.
model_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    api_key=api_key,
)


# Define a simple function tool that the agent can use.
# For this example, we use a fake weather tool for demonstration purposes.
async def get_weather(city: str) -> str:
    """Get the weather for a given city."""
    return f"The weather in {city} is 73 degrees and Sunny."


# Define an AssistantAgent with the model, tool, system message, and reflection enabled.
# The system message instructs the agent via natural language.
agent = AssistantAgent(
    name="weather_agent",
    model_client=model_client,
    tools=[get_weather],
    system_message="You are a helpful assistant.",
    reflect_on_tool_use=True,
    model_client_stream=True,  # Enable streaming tokens from the model client.
)


# Run the agent and stream the messages to the console.
async def main() -> None:
    await Console(agent.run_stream(task="What is the weather in New York?"))
    # Close the connection to the model client.
    await model_client.close()


# NOTE: if running this inside a Python script you'll need to use asyncio.run(main()).
await main() # if running from jupyter lab
asyncio.run(main()) # if running from outside terminal ==> py app.py